# HTU-inspired LWFA to beamline tutorial

This notebook analyzes the output of the two-stage tutorial:

1. a WarpX simulation of a self-injected, laser-wakefield accelerated
   electron bunch (`lwfa_warpx`);
2. that bunch propagated through the HTU beamline lattice with ImpactX
   (`beamline_impactx`).

Run this notebook from the `htu` directory, after both
`lwfa_warpx/run_lwfa_warpx.py` and `beamline_impactx/run_beamline_impactx.py`
(preceded by `beamline_impactx/warpx_to_beamline_impactx.py`) have completed.


In [ ]:
import glob
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import openpmd_api as io
import pandas as pd
from lwfa_warpx_utils import read_electrons

sys.path.insert(0, "lwfa_warpx")
sys.path.insert(0, "beamline_impactx")

## lwfa_warpx: the WarpX LWFA stage

We read the electron species from the last WarpX diagnostic dump and look
at its energy spectrum. The self-injected bunch should show up as a
distinct, quasi-monoenergetic population well above the cold background
plasma electrons, which stay near rest.


In [ ]:
x, y, z, ux, uy, uz, w, kinetic_energy_MeV = read_electrons("lwfa_warpx/diags/diag1")

fig, ax = plt.subplots()
ax.hist(kinetic_energy_MeV, bins=200, weights=w)
ax.set_xlabel("kinetic energy [MeV]")
ax.set_ylabel("number of electrons (weighted)")
ax.set_yscale("log")
ax.set_title("WarpX electron energy spectrum (final iteration)")
plt.show()

### Isolate the injected bunch

The energy cut below should match the `--energy-cut-MeV` used when running
`warpx_to_beamline_impactx.py` for `beamline_impactx`. Adjust it to sit just
above the cold background peak and below the injected bunch, based on the
histogram above.


In [ ]:
energy_cut_MeV = 20.0

mask = kinetic_energy_MeV > energy_cut_MeV
print(f"{mask.sum()} macroparticles above {energy_cut_MeV} MeV")

bunch_charge_pC = w[mask].sum() * 1.602176634e-19 * 1e12
mean_energy_MeV = np.average(kinetic_energy_MeV[mask], weights=w[mask])
std_energy_MeV = np.sqrt(
    np.average((kinetic_energy_MeV[mask] - mean_energy_MeV) ** 2, weights=w[mask])
)
print(f"bunch charge: {bunch_charge_pC:.3f} pC")
print(f"mean kinetic energy: {mean_energy_MeV:.2f} MeV")
print(f"energy spread (rms/mean): {100 * std_energy_MeV / mean_energy_MeV:.1f}%")

fig, ax = plt.subplots()
ax.hist(kinetic_energy_MeV[mask], bins=100, weights=w[mask])
ax.set_xlabel("kinetic energy [MeV]")
ax.set_ylabel("number of electrons (weighted)")
ax.set_title("Self-injected bunch spectrum")
plt.show()

fig, ax = plt.subplots()
sc = ax.scatter(z[mask] * 1e6, x[mask] * 1e6, c=kinetic_energy_MeV[mask], s=2)
ax.set_xlabel("z [um]")
ax.set_ylabel("x [um]")
ax.set_title("Injected bunch, longitudinal-horizontal phase space")
fig.colorbar(sc, label="kinetic energy [MeV]")
plt.show()

## beamline_impactx: the HTU beamline with ImpactX

ImpactX writes reduced beam moments (`sig_x`, `sig_y`, emittances, etc.) at
every lattice element to `beamline_impactx/diags/reduced_beam_characteristics.*`,
and full phase-space dumps at each `BeamMonitor` to
`beamline_impactx/diags/openPMD/<name>.h5`.


In [ ]:
def read_reduced_beam_characteristics(file_pattern):
    frames = []
    for filename in glob.glob(file_pattern):
        df = pd.read_csv(filename, delimiter=r"\s+")
        if "step" not in df.columns:
            step = int(re.findall(r"[0-9]+", filename)[0])
            df["step"] = step
        frames.append(df)
    return pd.concat(frames, axis=0, ignore_index=True).sort_values("s")


rbc = read_reduced_beam_characteristics(
    "beamline_impactx/diags/reduced_beam_characteristics.*"
)
rbc.head()

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8, 6))

axes[0].plot(rbc["s"], rbc["sig_x"] * 1e3, label=r"$\sigma_x$")
axes[0].plot(rbc["s"], rbc["sig_y"] * 1e3, label=r"$\sigma_y$")
axes[0].set_ylabel("beam size [mm]")
axes[0].legend()

axes[1].plot(rbc["s"], rbc["emittance_xn"] * 1e6, label=r"$\epsilon_{n,x}$")
axes[1].plot(rbc["s"], rbc["emittance_yn"] * 1e6, label=r"$\epsilon_{n,y}$")
axes[1].set_xlabel("s [m]")
axes[1].set_ylabel("normalized emittance [um]")
axes[1].legend()

fig.suptitle("Beam envelope through the HTU beamline")
plt.show()

### Phase space at a diagnostic screen

Each `BeamMonitor` in the lattice writes an openPMD file named after it.
The example below reads the beam at the entrance monitor (`monitor`, added
just before the first lattice element in `run_beamline_impactx.py`) and at
the last undulator screen, `UC_VisaEBeam8`.


In [ ]:
def read_screen(name, diags_dir="beamline_impactx/diags/openPMD"):
    series = io.Series(f"{diags_dir}/{name}.h5", io.Access.read_only)
    last_step = list(series.iterations)[-1]
    return series.iterations[last_step].particles["beam"].to_df()


initial = read_screen("monitor")
final = read_screen("UC_VisaEBeam8")

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
axes[0].scatter(initial["position_x"] * 1e3, initial["position_y"] * 1e3, s=2)
axes[0].set_title("Entrance to the HTU beamline")
axes[1].scatter(final["position_x"] * 1e3, final["position_y"] * 1e3, s=2)
axes[1].set_title("UC_VisaEBeam8 (end of undulator)")
for ax in axes:
    ax.set_xlabel("x [mm]")
    ax.set_ylabel("y [mm]")
plt.tight_layout()
plt.show()